# 01 — Data Prep
DAPT corpus chunking + SFT instruction dataset build. Run this first — everything downstream (`02`, `03`) reads from `data/processed/`.

**Runtime:** Runtime → Change runtime type → **T4 GPU** (chunking itself is CPU-bound, but keep GPU on so `02` can run in this same session without a restart).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/legal-compliance-slm"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
!git clone https://github.com/Shankar-behera/legal-compliance-slm.git
%cd legal-compliance-slm
!pip install -r requirements.txt -q

## DAPT corpus — pull from Hugging Face (pile-of-law)

No manual sourcing needed: this streams real regulatory/legal text from `pile-of-law/pile-of-law` on the Hub directly into `data/raw/dapt/*.txt`. Default subsets (`cfr`, `eurlex`, `privacy_policies`, `tos`) are the ones most relevant to a compliance auditor — see `data/scripts/download_corpus.py` for the full list of available subsets if you want different coverage.

In [ ]:
!python data/scripts/download_corpus.py \
    --subsets cfr eurlex privacy_policies tos \
    --max_docs_per_subset 1500 \
    --output_dir data/raw/dapt

## Chunk the DAPT corpus

In [ ]:
!python data/scripts/chunk_text.py \
    --input_dir data/raw/dapt \
    --output_path data/processed/dapt_chunks.jsonl \
    --tokenizer Qwen/Qwen2.5-1.5B-Instruct \
    --block_size 512

## Pull LexGLUE (unfair_tos) as a supplementary SFT source

Reformats LexGLUE's clause-level unfair-ToS labels into this project's scenario/clause_violation/remediation schema and writes them straight into `data/raw/sft/`, where the cell below picks them up alongside any synthetic scenario files.

In [ ]:
!python data/scripts/build_sft_from_lexglue.py \
    --output_path data/raw/sft/lexglue_scenarios.json \
    --max_examples 4000 \
    --include_fair_examples

## Build the SFT instruction dataset

In [ ]:
!python data/scripts/build_sft_dataset.py \
    --input_dir data/raw/sft \
    --output_path data/processed/sft_pairs.jsonl

## Dataset statistics

Measured counts for the README/model card — not the target volumes, the actual ones.

In [ ]:
!python -m data.scripts.dataset_stats \
    --dapt_path data/processed/dapt_chunks.jsonl \
    --sft_path data/processed/sft_pairs.jsonl \
    --tokenizer Qwen/Qwen2.5-1.5B-Instruct \
    --output_path docs/dataset_stats.md

## Persist processed data to Drive

So `02`/`03` can start a fresh runtime without redoing this step.

In [ ]:
import shutil
os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)
shutil.copy("data/processed/dapt_chunks.jsonl", f"{DRIVE_ROOT}/data/processed/dapt_chunks.jsonl")
shutil.copy("data/processed/sft_pairs.jsonl", f"{DRIVE_ROOT}/data/processed/sft_pairs.jsonl")
print("Copied processed data to Drive.")